In [1]:
import os
HOME = os.getcwd()
print(HOME)


%cd {HOME}
!git clone https://github.com/ultralytics/ultralytics.git
%cd {HOME}/ultralytics

import sys
sys.path.append(f"{HOME}/ultralytics")

/content
/content
Cloning into 'ultralytics'...
remote: Enumerating objects: 28876, done.
remote: Counting objects: 100% (588/588), done.
remote: Compressing objects: 100% (350/350), done.
remote: Total 28876 (delta 325), reused 420 (delta 237), pack-reused 28288
Receiving objects: 100% (28876/28876), 16.77 MiB | 8.91 MiB/s, done.
Resolving deltas: 100% (20338/20338), done.
/content/ultralytics


### **Load YOLOV8 models**

Download weights

In [6]:
%cd {HOME}/ultralytics
#!wget https://github.com/ultralytics/ultralytics.git/releases/download/v0.1/yolov7-e6e.pt --quiet
!wget /content/drive/MyDrive/Penalty_project/yolov8x-pose.pt --quiet

#DETECTION_MODEL_WEIGHTS_PATH = f"{HOME}/yolov9/yolov9-E.pt"
POSE_MODEL_WEIGHTS_PATH = f"{HOME}/drive/MyDrive/Penalty_project/yolov8x-pose.pt"



/content/ultralytics


In [3]:
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Load detection model

In [4]:
from ultralytics import YOLO
import pandas as pd

# Chargement du modèle YOLO
model = YOLO('yolov8x-pose.pt')

100%|██████████| 133M/133M [00:00<00:00, 259MB/s]


In [ ]:
# Chemin vers la source vidéo
video_path = "/content/drive/MyDrive/Penalty_project/My_Dataset/Final_set_25/penalty_76.mp4"
video_name = os.path.basename(video_path)

# Exécution du modèle sur la vidéo
results = model(source=video_path, show=True, conf=0.03, save=True, augment=False, max_det=3, retina_masks=True)


In [ ]:
def f(x,y) :
  # Chemin vers la source vidéo
  video_path = x
  video_name = os.path.basename(video_path)

  # Exécution du modèle sur la vidéo
  results = model(source=video_path, show=False, conf=0.15, save=False, augment=False, max_det=4, retina_masks=False)

  # Définir le répertoire de sortie basé sur le nom de la vidéo
  output_dir = os.path.join(y, os.path.splitext(video_name)[0])
  os.makedirs(output_dir, exist_ok=True)

  # Filtrer pour obtenir seulement le joueur le plus bas
  for index, result in enumerate(results):
      if result.boxes.data.shape[0] > 0:  # S'assurer qu'il y a des boîtes détectées
          # y2 est à l'index 3 si la structure est [x1, y1, x2, y2, conf, cls]
          y_max = result.boxes.data[:, 3]  # Obtenir toutes les coordonnées y2
          lowest_box_index = torch.argmax(y_max).item()  # Obtenir l'index de la boîte la plus basse
          #print(lowest_box_index)
      else:
          print(f"No boxes detected in frame {index}.")

          # Pour chaque keypoints
      if result.keypoints and len(result.keypoints) > lowest_box_index:
          keypoints = result.keypoints[lowest_box_index]
          keypoints_data = []
          for i in range(5, 17) :
            #print("Structure des keypoints :", keypoints.shape)  # Afficher la forme du tenseur
            keypoints_data.append({
                  'x': keypoints.xy[0][i][0].numpy(), # Convertir tensor en numpy array
                  'y': keypoints.xy[0][i][1].numpy(),
                  'z': keypoints.conf[0][i].numpy(),
                })

          # Vérifier le contenu de keypoints_data avant de créer le DataFrame
          #print("Données des keypoints :", keypoints_data)

          # Créer un DataFrame à partir des données de keypoints
          df_keypoints = pd.DataFrame(keypoints_data)


          # Générer un nom de fichier unique pour le CSV
          csv_filename = f"keypoints_frame_{index}.csv"
          csv_file = os.path.join(output_dir, csv_filename)

          # Exporter en CSV
          df_keypoints.to_csv(csv_file, index=True)
          #print(f"Saved keypoints for the lowest person to {csv_file}")
      else:
          print(f"No keypoints found for frame {index} or incorrect index reference.")



def f_0(x,y):
  for index in range(0,25):
    keypoints = []
    for i in range(5,17):
      keypoints.append({
          'x':0.0,
          'y':0.0,
          'z':0.0,
      })

    video_path = x
    video_name = os.path.basename(video_path)
    # Définir le répertoire de sortie basé sur le nom de la vidéo
    output_dir = os.path.join(y, os.path.splitext(video_name)[0])
    os.makedirs(output_dir, exist_ok=True)
    # Créer un DataFrame à partir des données de keypoints
    df_keypoints = pd.DataFrame(keypoints)

    # Générer un nom de fichier unique pour le CSV
    csv_filename = f"keypoints_frame_{index}.csv"
    csv_file = os.path.join(output_dir, csv_filename)

    # Exporter en CSV
    df_keypoints.to_csv(csv_file, index=True)

Apply the pose estimation on all the videos of a given directory and store the results in a given output

In [ ]:
chemin_dossier = '/content/drive/MyDrive/Penalty_project/My_Dataset/Final_set_25'
chemin_output = '/content/drive/MyDrive/Penalty_project/My_Dataset/output_2'

for fichier in os.listdir(chemin_dossier):
    if fichier.endswith('.mp4'):
        nom_dossier_sortie = fichier[:-4]  # Retirer l'extension .mp4 pour obtenir le nom du dossier
        chemin_complet = os.path.join(chemin_dossier, fichier)
        chemin_sortie = os.path.join(chemin_output, nom_dossier_sortie)

        if not os.path.exists(chemin_sortie):  # Vérifiez si le fichier de sortie existe déjà
            print(f'Traitement de la vidéo: {chemin_complet}')
            f(chemin_complet, chemin_output)
        else:
            print(f'La vidéo {chemin_complet} a déjà été traitée.')


La vidéo /content/drive/MyDrive/Penalty_project/My_Dataset/Final_set_25/penalty_61.mp4 a déjà été traitée.
La vidéo /content/drive/MyDrive/Penalty_project/My_Dataset/Final_set_25/penalty_62.mp4 a déjà été traitée.
La vidéo /content/drive/MyDrive/Penalty_project/My_Dataset/Final_set_25/penalty_63.mp4 a déjà été traitée.
La vidéo /content/drive/MyDrive/Penalty_project/My_Dataset/Final_set_25/penalty_64.mp4 a déjà été traitée.
La vidéo /content/drive/MyDrive/Penalty_project/My_Dataset/Final_set_25/penalty_65.mp4 a déjà été traitée.
La vidéo /content/drive/MyDrive/Penalty_project/My_Dataset/Final_set_25/penalty_66.mp4 a déjà été traitée.
La vidéo /content/drive/MyDrive/Penalty_project/My_Dataset/Final_set_25/penalty_67.mp4 a déjà été traitée.
La vidéo /content/drive/MyDrive/Penalty_project/My_Dataset/Final_set_25/penalty_68.mp4 a déjà été traitée.
La vidéo /content/drive/MyDrive/Penalty_project/My_Dataset/Final_set_25/penalty_69.mp4 a déjà été traitée.
La vidéo /content/drive/MyDrive/Penal